In [1]:
import sys
import importlib
from pathlib import Path

current = Path.cwd()

PROJECT_ROOT = None

for path in [current] + list(current.parents):
    if (path / "src").is_dir():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root containing 'src'."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

print(
    "Project root:",
    PROJECT_ROOT
)

import pandas as pd
import yaml

from src.fusion.registry_loader import (
    load_disease_registry,
)

import src.fusion.staged_fusion as staged_fusion

staged_fusion = importlib.reload(
    staged_fusion
)

StagedFusionEngine = (
    staged_fusion.StagedFusionEngine
)

print("Imports successful.")

Project root: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service
Imports successful.


In [2]:
REGISTRY_PATH = (
    PROJECT_ROOT
    / "config"
    / "disease_registry.yaml"
)

registry_config = load_disease_registry(
    REGISTRY_PATH
)

DISEASE_REGISTRY = (
    registry_config["diseases"]
)

GLOBAL_RULES = (
    registry_config.get(
        "global_rules",
        {}
    )
)

print(
    "Registry loaded:",
    REGISTRY_PATH
)

print(
    "Diseases:",
    len(DISEASE_REGISTRY)
)

Registry loaded: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\config\disease_registry.yaml
Diseases: 26


In [3]:
registry_rows = []

for disease, config in DISEASE_REGISTRY.items():

    registry_rows.append({
        "disease": disease,
        "display_name": config.get(
            "display_name",
            disease
        ),
        "category": config.get(
            "category"
        ),
        "status": config.get(
            "status"
        ),
        "primary_modality": config.get(
            "primary_modality"
        ),
        "primary_dataset": config.get(
            "primary_dataset"
        ),
        "confirmatory_modalities": ", ".join(
            config.get(
                "confirmatory_modalities",
                []
            )
        ),
    })

registry_df = pd.DataFrame(
    registry_rows
)

display(
    registry_df
)

,disease,display_name,category,status,primary_modality,primary_dataset,confirmatory_modalities
0,acute_mi,Acute Myocardial Infarction,coronary_ischemic,supported,biomarkers,zheen,ecg
1,myocardial_ischemia,Myocardial Ischemia,coronary_ischemic,supported_weaker,ecg,ptbxl,
2,st_t_abnormalities,ST/T Abnormalities,coronary_ischemic,supported,ecg,ptbxl,
3,atrial_fibrillation,Atrial Fibrillation,electrical_arrhythmic,supported,ecg,ptbxl,
4,vt_vf,Ventricular Tachycardia / Ventricular Fibrilla...,electrical_arrhythmic,unsupported,None,None,
5,bradyarrhythmia,Bradyarrhythmia,electrical_arrhythmic,supported,ecg,ptbxl,
6,av_block,Atrioventricular Block,electrical_arrhythmic,supported,ecg,ptbxl,
7,lbbb_rbbb,Left / Right Bundle Branch Block,electrical_arrhythmic,supported,ecg,ptbxl,
8,other_conduction_abnormalities,Other Conduction Abnormalities,electrical_arrhythmic,supported,ecg,ptbxl,
9,lvh_rvh,Left / Right Ventricular Hypertrophy,structural_functional,supported_weaker,ecg,ptbxl,


In [4]:
fusion_engine = StagedFusionEngine(
    DISEASE_REGISTRY
)

print(
    "Fusion engine ready."
)

Fusion engine ready.


In [5]:
VALIDATION_AUROC_PATHS = {
    "biomarkers":
        PROJECT_ROOT
        / "artifacts"
        / "metrics"
        / "biomarkers"
        / "biomarker_validation_metrics.csv",

    "ecg":
        PROJECT_ROOT
        / "artifacts"
        / "metrics"
        / "ecg"
        / "ptbxl_validation_metrics.csv",
}

In [6]:
for model_name, path in VALIDATION_AUROC_PATHS.items():

    if path.exists():
        print(
            f"{model_name}: found"
        )
    else:
        print(
            f"{model_name}: missing -> {path}"
        )

biomarkers: missing -> d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\metrics\biomarkers\biomarker_validation_metrics.csv
ecg: missing -> d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\metrics\ecg\ptbxl_validation_metrics.csv


In [7]:
prediction_paths = {
    "biomarkers": PROJECT_ROOT / "artifacts" / "metrics" / "biomarkers" / "biomarker_val_predictions.csv",
    "ecg": PROJECT_ROOT / "artifacts" / "metrics" / "ecg" / "ptbxl_val_predictions.csv",
    "echo": PROJECT_ROOT / "artifacts" / "metrics" / "echo" / "echonet_val_predictions.csv",
}

missing_required_paths = [
    prediction_paths["biomarkers"],
    prediction_paths["echo"],
]
missing_required_paths = [
    path for path in missing_required_paths if not path.exists()
]

if missing_required_paths:
    raise FileNotFoundError(
        "Missing required validation predictions: "
        + ", ".join(str(path) for path in missing_required_paths)
    )

biomarker_predictions = pd.read_csv(prediction_paths["biomarkers"])
echo_predictions = pd.read_csv(prediction_paths["echo"])

if not {"dataset", "target", "true", "prediction"}.issubset(biomarker_predictions.columns):
    raise ValueError("Unexpected biomarker validation prediction schema.")
if not {"true_EF", "predicted_EF"}.issubset(echo_predictions.columns):
    raise ValueError("Unexpected Echo validation prediction schema.")

real_predictions = {"biomarkers": {}, "ecg": {}, "echo": {}}

for _, row in biomarker_predictions.iterrows():
    real_predictions["biomarkers"].setdefault(
        row["target"],
        [],
    ).append(
        float(row["prediction"])
    )

real_predictions["echo"]["EF"] = float(
    echo_predictions["predicted_EF"].mean()
)

if prediction_paths["ecg"].exists():
    ecg_predictions = pd.read_csv(prediction_paths["ecg"])

    for column in ["NORM", "MI", "STTC", "CD", "HYP"]:
        probability_column = f"{column}_prob"
        if probability_column in ecg_predictions:
            real_predictions["ecg"][column] = float(
                ecg_predictions[probability_column].mean()
            )

    print("Loaded ECG validation predictions.")
else:
    print(
        "ECG validation predictions were not found; "
        "continuing without ECG evidence."
    )

print("Loaded real validation predictions from evaluation artifacts.")

blood_only = {
    "biomarkers": {
        "acute_mi": real_predictions["biomarkers"]["acute_mi"][0]
    }
}

result = fusion_engine.evaluate_disease(
    "acute_mi",
    blood_only,
)

print(result)

Loaded ECG validation predictions.
Loaded real validation predictions from evaluation artifacts.
{'disease': 'acute_mi', 'display_name': 'Acute Myocardial Infarction', 'risk': 0.3104313015937805, 'confidence': 'LOW', 'status': 'primary_only', 'detected_from': ['biomarkers'], 'recommendations': ['Upload ecg for confirmation.']}


In [8]:
blood_ecg_agreement = {
    "biomarkers": {
        "acute_mi": real_predictions["biomarkers"]["acute_mi"][0]
    }
}

if "MI" in real_predictions["ecg"]:
    blood_ecg_agreement["ecg"] = {
        "MI": real_predictions["ecg"]["MI"]
    }
else:
    print("ECG MI evidence unavailable; evaluating biomarkers only.")

result = fusion_engine.evaluate_disease(
    "acute_mi",
    blood_ecg_agreement
)

print(result)

{'disease': 'acute_mi', 'display_name': 'Acute Myocardial Infarction', 'risk': 0.3104313015937805, 'confidence': 'MODERATE', 'status': 'agreement', 'detected_from': ['biomarkers'], 'recommendations': [], 'confirmatory_evidence': {'ecg': 0.3965869256072413}, 'confirmation_message': 'ecg provides supporting evidence.'}


In [9]:
blood_ecg_disagreement = {
    "biomarkers": {
        "acute_mi": real_predictions["biomarkers"]["acute_mi"][0]
    }
}

if "MI" in real_predictions["ecg"]:
    blood_ecg_disagreement["ecg"] = {
        "MI": real_predictions["ecg"]["MI"]
    }
else:
    print("ECG MI evidence unavailable; evaluating biomarkers only.")

result = fusion_engine.evaluate_disease(
    "acute_mi",
    blood_ecg_disagreement
)

print(result)

{'disease': 'acute_mi', 'display_name': 'Acute Myocardial Infarction', 'risk': 0.3104313015937805, 'confidence': 'MODERATE', 'status': 'agreement', 'detected_from': ['biomarkers'], 'recommendations': [], 'confirmatory_evidence': {'ecg': 0.3965869256072413}, 'confirmation_message': 'ecg provides supporting evidence.'}


In [10]:
hfref_only = {
    "echo": {
        "EF": real_predictions["echo"]["EF"]
    }
}

result = fusion_engine.evaluate_disease(
    "hfref",
    hfref_only
)

print(result)

{'disease': 'hfref', 'display_name': 'HFrEF / Reduced LV Function', 'risk': 0.0, 'confidence': 'LOW', 'status': 'primary_only', 'detected_from': ['echo'], 'recommendations': ['Upload biomarkers for confirmation.']}


In [11]:
result = fusion_engine.evaluate_disease(
    "hfref",
    {
        "echo": {
            "EF": real_predictions["echo"]["EF"]
        }
    }
)

print(result)

{'disease': 'hfref', 'display_name': 'HFrEF / Reduced LV Function', 'risk': 0.0, 'confidence': 'LOW', 'status': 'primary_only', 'detected_from': ['echo'], 'recommendations': ['Upload biomarkers for confirmation.']}


In [12]:
result = fusion_engine.evaluate_disease(
    "vt_vf",
    {}
)

print(
    result
)

{'disease': 'vt_vf', 'display_name': 'Ventricular Tachycardia / Ventricular Fibrillation', 'risk': None, 'confidence': 'LOW', 'status': 'unsupported', 'detected_from': [], 'recommendations': [], 'message': 'No sufficient general-population VT/VF ground truth in the selected datasets.'}


In [13]:
result = fusion_engine.evaluate_disease(
    "acute_mi",
    {
        "biomarkers": {
            "acute_mi": real_predictions["biomarkers"]["acute_mi"][0]
        }
    }
)

print(result)

{'disease': 'acute_mi', 'display_name': 'Acute Myocardial Infarction', 'risk': 0.3104313015937805, 'confidence': 'LOW', 'status': 'primary_only', 'detected_from': ['biomarkers'], 'recommendations': ['Upload ecg for confirmation.']}


In [14]:
result = fusion_engine.evaluate_disease(
    "acute_mi",
    {
        "biomarkers": {
            "acute_mi": real_predictions["biomarkers"]["acute_mi"][0]
        }
    }
)

print(result)

{'disease': 'acute_mi', 'display_name': 'Acute Myocardial Infarction', 'risk': 0.3104313015937805, 'confidence': 'LOW', 'status': 'primary_only', 'detected_from': ['biomarkers'], 'recommendations': ['Upload ecg for confirmation.']}


In [15]:
def _scalarize(value):
    if isinstance(value, list):
        if not value:
            return None
        return float(pd.Series(value).mean())
    return value


evaluation_predictions = {
    modality: {
        target: _scalarize(value)
        for target, value in predictions.items()
        if value is not None
    }
    for modality, predictions in real_predictions.items()
}

all_results = fusion_engine.evaluate_all(
    evaluation_predictions
)

rows = []
for disease, result in all_results.items():
    rows.append({
        "disease": disease,
        "display_name": result.get("display_name"),
        "risk": result.get("risk"),
        "confidence": result.get("confidence"),
        "status": result.get("status"),
    })

fusion_results_df = pd.DataFrame(rows)
display(fusion_results_df)

,disease,display_name,risk,confidence,status
0,acute_mi,Acute Myocardial Infarction,0.484861,MODERATE,agreement
1,myocardial_ischemia,Myocardial Ischemia,0.328239,LOW,primary_only
2,st_t_abnormalities,ST/T Abnormalities,0.328239,LOW,primary_only
3,atrial_fibrillation,Atrial Fibrillation,NaN,LOW,primary_missing
4,vt_vf,Ventricular Tachycardia / Ventricular Fibrilla...,NaN,LOW,unsupported
5,bradyarrhythmia,Bradyarrhythmia,NaN,LOW,primary_missing
6,av_block,Atrioventricular Block,NaN,LOW,primary_missing
7,lbbb_rbbb,Left / Right Bundle Branch Block,NaN,LOW,primary_missing
8,other_conduction_abnormalities,Other Conduction Abnormalities,NaN,LOW,primary_missing
9,lvh_rvh,Left / Right Ventricular Hypertrophy,0.348490,LOW,primary_only


In [16]:
biomarker_ecg_predictions = {
    "biomarkers": {
        "acute_mi": real_predictions["biomarkers"]["acute_mi"][0]
    }
}

if "MI" in real_predictions["ecg"]:
    biomarker_ecg_predictions["ecg"] = {
        "MI": real_predictions["ecg"]["MI"]
    }

scenarios = [
    {
        "scenario": "Biomarker and ECG validation outputs",
        "available": ", ".join(biomarker_ecg_predictions.keys()),
        "disease": "acute_mi",
        "predictions": biomarker_ecg_predictions,
    },
    {
        "scenario": "Echo validation output",
        "available": "echo",
        "disease": "hfref",
        "predictions": {
            "echo": {
                "EF": real_predictions["echo"]["EF"]
            },
        },
    },
    {
        "scenario": "All available validation outputs",
        "available": ", ".join(
            modality
            for modality, predictions in real_predictions.items()
            if predictions
        ),
        "disease": "acute_mi",
        "predictions": evaluation_predictions,
    },
]

scenario_results = []
for scenario in scenarios:
    result = fusion_engine.evaluate_disease(
        scenario["disease"],
        scenario["predictions"],
    )
    scenario_results.append({
        "scenario": scenario["scenario"],
        "available": scenario["available"],
        "disease": scenario["disease"],
        "risk": result.get("risk"),
        "confidence": result.get("confidence"),
        "status": result.get("status"),
        "recommendations": "; ".join(result.get("recommendations", [])),
    })

scenario_df = pd.DataFrame(scenario_results)
display(scenario_df)

,scenario,available,disease,risk,confidence,status,recommendations
0,Biomarker and ECG validation outputs,"biomarkers, ecg",acute_mi,0.310431,MODERATE,agreement,
1,Echo validation output,echo,hfref,0.000000,LOW,primary_only,Upload biomarkers for confirmation.
2,All available validation outputs,"biomarkers, ecg, echo",acute_mi,0.484861,MODERATE,agreement,


In [17]:
FUSION_RESULTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "fusion"
)

FUSION_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

scenario_path = (
    FUSION_RESULTS_DIR
    / "fusion_scenarios.csv"
)

scenario_df.to_csv(
    scenario_path,
    index=False
)

registry_path = (
    FUSION_RESULTS_DIR
    / "registry_snapshot.csv"
)

registry_df.to_csv(
    registry_path,
    index=False
)

print(
    "Saved scenario results:",
    scenario_path
)

print(
    "Saved registry snapshot:",
    registry_path
)

Saved scenario results: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\fusion\fusion_scenarios.csv
Saved registry snapshot: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\fusion\registry_snapshot.csv
